## **Modelamiento Supervisado - Línea Base de Clasificación**

**Integrantes:**
* Gino Andrades (`gi.andrades@duocuc.cl`)
* Yai Selti (`ya.selti@duocuc.cl`)
* Miguel Villarroel (`mig.villarroel@duocuc.cl`)

**Fecha de Creación:** 24 de Mayo de 2026  
**Versión:** 1.2  
**Institución:** Duoc UC

---

## **Descripción del Problema de Negocio**
El desafío crítico actual de la institución financiera radica en una disminución sostenida en la **tasa de retención de sus clientes (Churn)**, un fenómeno que impacta directamente el flujo de ingresos operativos y eleva drásticamente los costos asociados a la adquisición de nuevas cuentas de clientes.

El objetivo primordial de este desarrollo es estructurar una solución analítica avanzada que permita anticipar el comportamiento de abandono de los usuarios.

---

## **Requisitos de Software y Entorno**
Este ecosistema de modelamiento fue desarrollado y testeado utilizando el lenguaje **Python 3.12**. Para asegurar la correcta reproducibilidad de los pipelines estadísticos y la ejecución de la validación cruzada, se requiere la instalación de las siguientes bibliotecas base:

* **`pandas`** (>= 1.1.0) — *Estructuración tabular y manipulación de datos.*
* **`numpy`** (>= 2.0.2) — *Cómputo matricial y soporte algebraico.*
* **`scikit-learn`** (>= 1.3.0) — *Construcción de Pipelines, Imputación Iterativa y algoritmos supervisados.*
* **`matplotlib`** (>= 3.7.1) — *Motor gráfico para visualizaciones base.*
* **`seaborn`** (>= 0.13.2) — *Estructuras de visualización estadística complementaria.*
---
Para verificar la versión instalada ejecutar usando el siguiente comando, usando la librería de la cual quieres saber la versión:

In [103]:
import pandas as pd
print(pd.__version__)

3.0.3


## **1. Configuración del Entorno (Code)**

###*1.1 Importaciones necesarias para trabajar.*

In [104]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import sys
sys.path.append('..')

from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, OneHotEncoder, FunctionTransformer
# 1. Habilitar explícitamente las características experimentales
from sklearn.experimental import enable_iterative_imputer
from sklearn.impute import IterativeImputer
from sklearn.metrics import f1_score
# Aqui llamamos a nuestro archivo donde quedaron las funciones de limpieza y transformacion.
from src.data_preprocessing import tratar_duplicados, corregir_signos_negativos, ingenieria_caracteristicas, Winsorizer, DataFrameConverter, CorrelationFilter
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.svm import SVC
from sklearn.pipeline import Pipeline, make_pipeline
from sklearn.linear_model import LogisticRegression, BayesianRidge

from src.model_evaluation import evaluar_modelo_entrenado

### *1.2 Cargar los registros a la variable "data" con pandas*

In [105]:
ruta_dataset = "../data/data_sucia/dataset_clientes.csv"
data = pd.read_csv(ruta_dataset)

## **2. Preprocesamiento Inicial**

In [106]:
cols_con_negativos = ['ingreso_mensual', 'gasto_mensual', 'deuda_total']

pipeline_limpieza = Pipeline(steps=[
    ('limpieza_duplicados', FunctionTransformer(tratar_duplicados, kw_args={'drop': True})),
    ('correccion_signos', FunctionTransformer(corregir_signos_negativos, kw_args={'columnas': cols_con_negativos})),
    ('creacion_variables', FunctionTransformer(ingenieria_caracteristicas))
])


data_limpio = pipeline_limpieza.fit_transform(data)

X = data_limpio.drop(columns=['id_cliente', 'fecha_registro', 'dia_semana_registro', 'abandono'], errors='ignore')
y = data_limpio['abandono']

# División estratificada de control (Semilla 42)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.20, random_state=42, stratify=y)

# Todas las variables continuas, discretas, conteos y banderas binarias (0/1)
num_cols = [
    'edad',
    'ingreso_mensual',
    'gasto_mensual',
    'deuda_total',
    'score_crediticio',
    'antiguedad_meses',
    'frecuencia_compra',
    'ultima_compra_dias',
    'tiene_tarjeta_credito',
    'ratio_deuda',
    'ratio_gasto',
    'indice_lealtad',
    'riesgo_por_inactividad'
]

# Todas las variables cualitativas que pasarán por el OneHotEncoder
cat_cols = [
    'genero',
    'region',
    'uso_app',
    'tipo_plan',
]

## **3. Canalización Estadística (Pipelines de Transformación)**

In [107]:
# Creamos el estimador interno empaquetado con su propio escalador.
# Esto asegura que cada regresión interna reciba los predictores estandarizados (media 0, var 1),
# eliminando el sesgo por diferencia de escalas en la penalización de BayesianRidge.
estimador_interno = make_pipeline(
    StandardScaler(),
    BayesianRidge()
)

num_pipeline = Pipeline(steps=[
    ('imputacion_iterativa', IterativeImputer(estimator=estimador_interno, random_state=42, max_iter=10)),
    ('tratamiento_outliers', Winsorizer(limits=(0.01, 0.01))),
    ('escalado_estandar', StandardScaler())
])

cat_pipeline = Pipeline(steps=[
    ('codificacion_onehot', OneHotEncoder(handle_unknown='ignore', sparse_output=False))
])

preprocessor = ColumnTransformer(transformers=[
    ('num', num_pipeline, num_cols),
    ('cat', cat_pipeline, cat_cols)
], remainder='drop')

## **4. Modelos Supervisados**

### *4.1 Modelo de Regresión Logística*

In [108]:
resultados_base = {}

In [109]:
pipeline_lr_base = Pipeline(steps=[
    ('duplicados', FunctionTransformer(tratar_duplicados, kw_args={'drop': False})),
    ('preprocesador', preprocessor),
    ('conversion', DataFrameConverter(preprocessor)),
    ('colinealidad', CorrelationFilter(threshold=0.9)),
    ('clf', LogisticRegression(random_state=42, max_iter=1000, solver='saga'))
])
pipeline_lr_base.fit(X_train, y_train)
resultados_base['Regresión Logística'] = pipeline_lr_base

# Evaluar el modelo
mejores_metricas_lr = evaluar_modelo_entrenado(pipeline_lr_base, X_test, y_test)

print("Metricas de Regresión Logística")
for metrica, valor in mejores_metricas_lr.items():
    print(f"{metrica:<15} : {valor:.4f}")

Metricas de Regresión Logística
accuracy        : 0.6570
precision       : 0.6061
recall          : 0.3869
f1              : 0.4723
roc_auc         : 0.6883


**Evaluacion del Entrenamiento**

In [110]:
print(f"{'Score del modelo en entrenamiento':<40}:{pipeline_lr_base.score(X_train, y_train):.5f}")
print(f"{'Score del modelo en test': <40}:{pipeline_lr_base.score(X_test, y_test):.5f}")

Score del modelo en entrenamiento       :0.65075
Score del modelo en test                :0.65700


#### *4.1.1 Observación: Regresión Logística Línea Base*
* **Comportamiento:** El modelo exhibe un comportamiento estable entre sus conjuntos de datos, obteniendo un score de exactitud balanceado del ~65.0% tanto en entrenamiento como en prueba. Esto descarta la presencia de sobreajuste (overfitting).
* **Impacto en el Negocio (Churn):** A pesar de su estabilidad, el indicador de **Recall (Sensibilidad)** es críticamente bajo (~38.99%). Esto significa que el algoritmo actual es "ciego" ante el 61% de los clientes que efectivamente van a abandonar el banco, lo que resultaría en pérdidas financieras por falta de acción preventiva. Esta debilidad justifica la necesidad de una optimización de hiperparámetros y umbrales de decisión en las siguientes etapas.

### *4.2 Modelo de Árbol de Decisión*

In [111]:
pipeline_dt_base = Pipeline(steps=[
    ('duplicados', FunctionTransformer(tratar_duplicados, kw_args={'drop': False})),
    ('preprocesador', preprocessor),
    ('conversion', DataFrameConverter(preprocessor)),
    ('colinealidad', CorrelationFilter(threshold=0.9)),
    ('clf', DecisionTreeClassifier(random_state=42))
])
pipeline_dt_base.fit(X_train, y_train)
resultados_base['Árbol de Decisión'] = pipeline_dt_base

# Evaluar el modelo
mejores_metricas_lr = evaluar_modelo_entrenado(pipeline_dt_base, X_test, y_test)

print("Metricas del Árbol de Decisión")
for metrica, valor in mejores_metricas_lr.items():
    print(f"{metrica:<15} : {valor:.4f}")


Metricas del Árbol de Decisión
accuracy        : 0.5695
precision       : 0.4577
recall          : 0.4606
f1              : 0.4592
roc_auc         : 0.5509


**Evaluacion del Entrenamiento**

In [112]:
print(f"{'Score del modelo en entrenamiento':<40}:{pipeline_dt_base.score(X_train, y_train):.5f}")
print(f"{'Score del modelo en test': <40}:{pipeline_dt_base.score(X_test, y_test):.5f}")

Score del modelo en entrenamiento       :1.00000
Score del modelo en test                :0.56950


#### **4.1.2 Observación: Evidencia de Sobreajuste**
* **Overfitting:** El Árbol de Decisión muestra un resultado de **1.00000 (100% de precisión) en el conjunto de entrenamiento**, pero se desploma drásticamente a un **0.5695 (~56.9%) en el conjunto de prueba (test)**.
* **Explicación:** Al instanciarse el algoritmo con sus parámetros por defecto, la profundidad máxima (`max_depth`) queda libre. El árbol continuó ramificándose de manera infinita hasta memorizar individualmente cada registro de entrenamiento, incluyendo el ruido y las anomalías de los datos sucios.
* **Optimización:** Consideramos que seria lo mas optimo utilizar la tecnica de regularizacion **(poda/pruning)** mediante búsquedas en grilla (`GridSearchCV`) , limitando el crecimiento del árbol para forzarlo a generalizar patrones bancarios reales.

### *4.3 Modelo de Máquinas de Vectores de Soporte (SVM)*

In [113]:
pipeline_svm_base = Pipeline(steps=[
    ('duplicados', FunctionTransformer(tratar_duplicados, kw_args={'drop': False})),
    ('preprocesador', preprocessor),
    ('conversion', DataFrameConverter(preprocessor)),
    ('colinealidad', CorrelationFilter(threshold=0.9)),
    ('clf', SVC(random_state=42, kernel='rbf', probability=True))
])
pipeline_svm_base.fit(X_train, y_train)
resultados_base['SVM'] = pipeline_svm_base

# Evaluar el modelo
mejores_metricas_lr = evaluar_modelo_entrenado(pipeline_svm_base, X_test, y_test)

print("Metricas Maquinas de Vectores de Soporte")
for metrica, valor in mejores_metricas_lr.items():
    print(f"{metrica:<15} : {valor:.4f}")

Metricas Maquinas de Vectores de Soporte
accuracy        : 0.6448
precision       : 0.6064
recall          : 0.2980
f1              : 0.3997
roc_auc         : 0.6553


**Evaluacion del Entrenamiento**

In [114]:
print(f"{'Score del modelo en entrenamiento':<40}:{pipeline_svm_base.score(X_train, y_train):.5f}")
print(f"{'Score del modelo en test': <40}:{pipeline_svm_base.score(X_test, y_test):.5f}")

Score del modelo en entrenamiento       :0.68100
Score del modelo en test                :0.64475


#### **4.1.3 Observación: Máquinas de Vectores de Soporte (SVM)**

* **Diagnóstico Estadístico:** El modelo SVM base no presenta sobreajuste (overfitting), ya que la diferencia entre entrenamiento (0.681) y prueba (0.645) es mínima. Sin embargo, padece de subajuste (underfitting) debido a la rigidez de su frontera de decisión.

* **Impacto en el Negocio:** El modelo es inviable operativamente en este estado. Su Recall es de apenas 29.8%, lo que significa que el banco dejaría escapar al 70.2% de los clientes que se van a fugar sin activar ninguna alerta. El F1-Score resultante es de 0.39%.

* **Estrategia de Optimización:** Este rendimiento plano justifica la necesidad de usar GridSearchCV. Modificando los hiperparámetros por defecto (C=1.0 y gamma='scale') por valores más agresivos (como C=10.0), se flexibilizará el hiperplano para capturar más clientes en riesgo.

## **5. Evaluación Comparativa de Métricas**

In [115]:
print("F1-SCORE MODELOS LINEA BASE")
for nombre, modelo in resultados_base.items():
    preds = modelo.predict(X_test)
    print(f"{nombre}: {f1_score(y_test, preds):.4f}")

F1-SCORE MODELOS LINEA BASE
Regresión Logística: 0.4723
Árbol de Decisión: 0.4592
SVM: 0.3997


In [116]:
# Variables que fueron eliminadas por presentar colinealidad
pipeline_lr_base.named_steps["colinealidad"].columns_to_drop_

[]

In [117]:
# Variables con las cuales fue entrenado el modelo
pipeline_lr_base.named_steps["conversion"].feature_names_

array(['num__edad', 'num__ingreso_mensual', 'num__gasto_mensual',
       'num__deuda_total', 'num__score_crediticio',
       'num__antiguedad_meses', 'num__frecuencia_compra',
       'num__ultima_compra_dias', 'num__tiene_tarjeta_credito',
       'num__ratio_deuda', 'num__ratio_gasto', 'num__indice_lealtad',
       'num__riesgo_por_inactividad', 'cat__genero_Femenino',
       'cat__genero_Masculino', 'cat__genero_Otro', 'cat__region_Centro',
       'cat__region_Norte', 'cat__region_Sur', 'cat__uso_app_Alto',
       'cat__uso_app_Bajo', 'cat__uso_app_Medio', 'cat__tipo_plan_Basico',
       'cat__tipo_plan_Estandar', 'cat__tipo_plan_Premium'], dtype=object)

#### **Análisis de Diagnóstico de Datos y Colinealidad**
* **Filtro de Correlación:** Al inspeccionar el paso final del pipeline, el atributo `columns_to_drop_` del `CorrelationFilter` arrojó una lista vacía `[]`. Esto significa que ninguna de las características predictoras superó el umbral crítico de multicolinealidad fijado en $0.90$.
* **Conclusión:** Este resultado valida la robustez de la Ingeniería de Características realizada previamente; las variables complejas y ratios financieros inyectados aportan información relevante y complementaria al modelo, enriqueciendo la capacidad predictiva sin introducir redundancia matemática en la matriz de covarianza.